# Capstone — Network-Traffic Anomaly Detection with Deep Learning

**The production app:** detect anomalous / malicious network traffic with a deep-learning **autoencoder** — trained on *normal* traffic, flagging anything that reconstructs poorly.

**The twist:** your *local*, fine-tuned model is the **copilot that writes the code**, running on your Mac via **Ollama** or via **Hugging Face Inference API** — your choice. The data never leaves your machine — exactly why local matters for security telemetry.

Steps: **clean the data → build & train the DNN (Keras) → measure accuracy (sklearn) → deploy.**

Read alongside [docs/10-capstone-build-a-production-dnn.md](../docs/10-capstone-build-a-production-dnn.md).

**Before you start:**
```bash
# Option A — local (default): serve the model on your Mac
ollama serve
ollama run granite-ml-coder "ready?"

# Option B — HF Inference: export these instead
#   export COPILOT_BACKEND=hf
#   export HF_TOKEN=hf_xxx
#   export HF_ENDPOINT_URL=https://<your-endpoint>.endpoints.huggingface.cloud/v1/chat/completions

uv pip install tensorflow scikit-learn pandas matplotlib requests joblib jupyter
```

## 0. Your local model as a copilot (Ollama or HF Inference)

This helper sends a prompt to your fine-tuned model. The **same model** answers whether it runs locally via Ollama or via the Hugging Face Inference API — flip `COPILOT_BACKEND`. We use it to *draft* code; you verify and run it. (At 1B it won't always be right — treat it as a fast first draft.)

In [ ]:
import os, requests

# Both backends serve the SAME fine-tuned model:
#   'ollama' -> local on your Mac          (default, fully offline)
#   'hf'     -> Hugging Face Inference API  (OpenAI-compatible endpoint)
BACKEND = os.environ.get("COPILOT_BACKEND", "ollama")

def ask(prompt: str) -> str:
    """Ask your fine-tuned model via Ollama (local) or HF Inference (cloud)."""
    msgs = [{"role": "user", "content": prompt}]
    if BACKEND == "hf":
        # HF Inference Endpoint (set HF_ENDPOINT_URL) or the serverless router.
        url = os.environ.get("HF_ENDPOINT_URL",
                             "https://router.huggingface.co/v1/chat/completions")
        token = os.environ["HF_TOKEN"]  # read/write token
        model = os.environ.get("HF_MODEL", "Tekimax/granite-ml-coder")
        r = requests.post(url, headers={"Authorization": f"Bearer {token}"},
                          json={"model": model, "messages": msgs, "stream": False},
                          timeout=180)
        r.raise_for_status()
        return r.json()["choices"][0]["message"]["content"]
    # default: local Ollama
    r = requests.post("http://localhost:11434/api/chat",
                      json={"model": os.environ.get("OLLAMA_MODEL", "granite-ml-coder"),
                            "messages": msgs, "stream": False}, timeout=180)
    r.raise_for_status()
    return r.json()["message"]["content"]

print(f"copilot backend: {BACKEND}")
print(ask("In one sentence, how does an autoencoder detect anomalies?"))

## 1. Ask the copilot to scaffold data cleaning

We read its suggestion, then write the verified version ourselves in the next cells.

In [ ]:
print(ask(
    "I have the KDD Cup 99 network intrusion dataset as a pandas DataFrame with "
    "categorical columns protocol_type, service, flag and a 'labels' column where "
    "'normal.' is benign. Write Python (pandas + scikit-learn) to: one-hot encode the "
    "categoricals, make a binary target (0=normal, 1=attack), and standardize features. "
    "Explain each step."
))

## 2. Load and clean the data

We use the built-in **KDD Cup '99** network-intrusion dataset (no manual download). Real network connections, labeled normal vs. many attack types. Set seeds for reproducibility.

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.datasets import fetch_kddcup99
from sklearn.preprocessing import StandardScaler

tf.random.set_seed(42)
np.random.seed(42)

# 10% subset is plenty and fast.
data = fetch_kddcup99(percent10=True, as_frame=True)
df = data.frame.copy()

# Decode byte-strings that the loader returns (b'tcp' -> 'tcp').
for col in df.columns:
    if df[col].dtype == object:
        df[col] = df[col].apply(lambda v: v.decode() if isinstance(v, bytes) else v)

print(df.shape)
print(df['labels'].value_counts().head())

In [ ]:
# Binary target: 0 = normal, 1 = attack (anything not 'normal.').
y = (df['labels'] != 'normal.').astype(int).values
X = df.drop(columns=['labels'])

# One-hot encode the 3 categorical columns; everything else is numeric.
X = pd.get_dummies(X, columns=['protocol_type', 'service', 'flag'])
X = X.astype('float32')
print('feature matrix:', X.shape, '| attack rate: %.2f%%' % (100 * y.mean()))

## 3. Split — train the autoencoder on NORMAL traffic only

Key idea: the model learns what *normal* looks like. We never show it attacks during training, so it generalizes to **novel / zero-day** anomalies. We scale using statistics from normal-train only (no leakage).

In [ ]:
from sklearn.model_selection import train_test_split

X_norm = X[y == 0].values
# Train/val from normal only; test = held-out normal + ALL attacks.
X_tr_norm, X_val_norm = train_test_split(X_norm, test_size=0.2, random_state=42)

X_attack = X[y == 1].values
X_test = np.vstack([X_val_norm, X_attack])
y_test = np.concatenate([np.zeros(len(X_val_norm)), np.ones(len(X_attack))])

# Scale on normal-train statistics only.
scaler = StandardScaler().fit(X_tr_norm)
X_tr_norm = scaler.transform(X_tr_norm)
X_val_norm = scaler.transform(X_val_norm)
X_test = scaler.transform(X_test)
print('train(normal):', X_tr_norm.shape, '| test(mixed):', X_test.shape)

## 4. Build the deep autoencoder (Keras)

Encoder compresses each connection to a small bottleneck; decoder reconstructs it. `Dropout` regularizes (fights overfitting); `adam` is an adaptive **gradient-descent** optimizer.

In [ ]:
input_dim = X_tr_norm.shape[1]

model = keras.Sequential([
    keras.layers.Input((input_dim,)),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.1),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(16, activation='relu'),   # bottleneck
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(input_dim, activation='linear'),  # reconstruction
])
model.compile(optimizer='adam', loss='mse')
model.summary()

## 5. Train (reconstruct normal traffic)

The target *is* the input — the network learns to rebuild normal connections. Early stopping restores the best weights when validation loss stops improving.

In [ ]:
early = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3,
                                      restore_best_weights=True)
history = model.fit(
    X_tr_norm, X_tr_norm,
    validation_data=(X_val_norm, X_val_norm),
    epochs=30, batch_size=256, callbacks=[early], verbose=2,
)

## 6. Reconstruction error → anomaly threshold

Normal traffic reconstructs with low error; attacks reconstruct poorly. We set the threshold from normal-validation errors (e.g. the 99th percentile).

In [ ]:
def recon_error(x):
    pred = model.predict(x, verbose=0)
    return np.mean(np.square(x - pred), axis=1)

val_err = recon_error(X_val_norm)
threshold = np.percentile(val_err, 99)   # tolerate ~1% false positives on normal
print(f'anomaly threshold = {threshold:.5f}')

## 7. Measure accuracy on the mixed test set (sklearn)

The honest evaluation: held-out normal + all attacks. Anything above threshold is flagged as an anomaly.

In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, accuracy_score)

test_err = recon_error(X_test)
y_pred = (test_err > threshold).astype(int)

print('Accuracy : %.4f' % accuracy_score(y_test, y_pred))
print('ROC-AUC  : %.4f' % roc_auc_score(y_test, test_err))
print()
print(classification_report(y_test, y_pred, target_names=['normal', 'attack']))
print('Confusion matrix [rows=true, cols=pred]:')
print(confusion_matrix(y_test, y_pred))

In [ ]:
import matplotlib.pyplot as plt

plt.hist(test_err[y_test == 0], bins=60, alpha=0.6, label='normal', log=True)
plt.hist(test_err[y_test == 1], bins=60, alpha=0.6, label='attack', log=True)
plt.axvline(threshold, color='k', linestyle='--', label='threshold')
plt.xlabel('reconstruction error'); plt.ylabel('count (log)'); plt.legend()
plt.title('Normal vs. attack reconstruction error')
plt.show()

## 8. Ask the copilot how to improve accuracy

When you plateau, let your model (local or HF) brainstorm — then *you* decide what to try.

In [ ]:
auc = roc_auc_score(y_test, test_err)
print(ask(
    f"My Keras autoencoder anomaly detector for network traffic gets ROC-AUC {auc:.3f}. "
    "Suggest 3 concrete changes (architecture, threshold, or features) to improve "
    "detection without too many false positives, and why each helps."
))

## 9. Deploy in production

Save the model, the scaler, and the threshold — the three things you need at inference — and wrap a clean `score()` API. This is what you'd put behind FastAPI, a stream processor, or a Cloudflare Worker.

In [ ]:
import json, joblib

model.save('netanomaly_autoencoder.keras')
joblib.dump(scaler, 'netanomaly_scaler.joblib')
json.dump({'threshold': float(threshold), 'columns': list(X.columns)},
          open('netanomaly_meta.json', 'w'))

def score(flows: np.ndarray) -> dict:
    """Production inference: raw feature rows -> anomaly flag + score."""
    m = keras.models.load_model('netanomaly_autoencoder.keras')
    sc = joblib.load('netanomaly_scaler.joblib')
    meta = json.load(open('netanomaly_meta.json'))
    x = sc.transform(flows)
    err = np.mean(np.square(x - m.predict(x, verbose=0)), axis=1)
    return {'anomaly': (err > meta['threshold']).tolist(), 'score': err.tolist()}

# Smoke test the deployed surface on a few raw attack rows.
print(score(X[y == 1].values[:3]))

## Done 🎉

You shipped a production-ready network-anomaly detector, built with your own model as copilot — local (Ollama) **or** via HF Inference:

- **Private** — traffic data never left the machine
- **Accurate** — high ROC-AUC separating attacks from normal
- **Deployable** — `netanomaly_autoencoder.keras` + `scaler` + `threshold` + a `score()` API
- **Governed** — versioned model, scaler, and threshold = reproducible, auditable lineage

Swap in your own traffic logs (NetFlow / Zeek / packet features) and the loop is identical. That's the workshop: **own your model, host it locally or on HF, and ship real ML with it.**